---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！

# 03 - DataLoader 使用

## 学习目标

1. 理解 DataLoader 的作用
2. 掌握 DataLoader 的重要参数
3. 学会批量加载数据
4. 理解 collate_fn 的使用场景

## 1. 为什么需要 DataLoader？

Dataset 只能一次返回一个样本，但训练神经网络需要：

- **批量处理**：一次处理多个样本（batch），提高 GPU 利用率
- **打乱数据**：每个 epoch 打乱数据顺序，避免模型记住顺序
- **并行加载**：多进程预加载数据，避免 GPU 等待
- **自动拼接**：将多个样本自动拼接成 batch tensor

DataLoader 就是为了解决这些问题！

## 2. DataLoader 的工作原理

```
Dataset: [样本0, 样本1, 样本2, 样本3, 样本4, 样本5, 样本6, 样本7, 样本8, 样本9]
         ↓ (batch_size=3, shuffle=False)
DataLoader:
  Batch 0: [样本0, 样本1, 样本2]  → shape: [3, ...]
  Batch 1: [样本3, 样本4, 样本5]  → shape: [3, ...]
  Batch 2: [样本6, 样本7, 样本8]  → shape: [3, ...]
  Batch 3: [样本9]                → shape: [1, ...] (最后一个batch)
```

## 3. 基础示例：理解批量加载

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# 创建简单的数据集
class NumberDataset(Dataset):
    def __init__(self, n):
        self.data = list(range(n))
    
    def __getitem__(self, idx):
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

# 创建数据集：0-9 共 10 个数字
dataset = NumberDataset(10)

# 创建 DataLoader
loader = DataLoader(
    dataset=dataset,
    batch_size=3,
    shuffle=False
)

print("查看每个 batch：\n")
for batch_idx, batch_data in enumerate(loader):
    print(f"Batch {batch_idx}: {batch_data}")
    print(f"  形状: {batch_data.shape}")
    print(f"  类型: {type(batch_data)}\n")

**关键观察：**

1. DataLoader 自动将数据分成多个 batch
2. 每个 batch 是一个 tensor（自动转换）
3. 最后一个 batch 可能不完整（只有 1 个元素）

## 4. 重要参数详解

### 4.1 batch_size：批量大小

In [ ]:
dataset = NumberDataset(10)

# batch_size=2
loader_bs2 = DataLoader(dataset, batch_size=2)
print("batch_size=2:")
for batch in loader_bs2:
    print(f"  {batch}")

# batch_size=5
loader_bs5 = DataLoader(dataset, batch_size=5)
print("\nbatch_size=5:")
for batch in loader_bs5:
    print(f"  {batch}")

### 4.2 shuffle：是否打乱数据

In [ ]:
dataset = NumberDataset(10)

# shuffle=False（默认）
loader_no_shuffle = DataLoader(dataset, batch_size=3, shuffle=False)
print("shuffle=False（顺序加载）:")
for batch in loader_no_shuffle:
    print(f"  {batch}")

# shuffle=True
loader_shuffle = DataLoader(dataset, batch_size=3, shuffle=True)
print("\nshuffle=True（随机打乱）:")
for batch in loader_shuffle:
    print(f"  {batch}")

print("\n再次遍历（顺序不同）:")
for batch in loader_shuffle:
    print(f"  {batch}")

**使用建议：**

- 训练集：`shuffle=True` （提高泛化能力）
- 验证集/测试集：`shuffle=False` （便于复现结果）

### 4.3 drop_last：是否丢弃最后不完整的 batch

In [ ]:
dataset = NumberDataset(10)

# drop_last=False（默认）：保留最后不完整的 batch
loader_keep = DataLoader(dataset, batch_size=3, drop_last=False)
print("drop_last=False（保留最后的 batch）:")
for i, batch in enumerate(loader_keep):
    print(f"  Batch {i}: {batch}, 大小: {len(batch)}")

# drop_last=True：丢弃最后不完整的 batch
loader_drop = DataLoader(dataset, batch_size=3, drop_last=True)
print("\ndrop_last=True（丢弃最后的 batch）:")
for i, batch in enumerate(loader_drop):
    print(f"  Batch {i}: {batch}, 大小: {len(batch)}")

**何时使用 drop_last=True？**

- 使用 Batch Normalization 时（BN 在 batch_size=1 时会出问题）
- 需要所有 batch 大小一致时

**一般情况下使用默认值 False 即可。**

## 5. 处理图像数据

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

class MockImageDataset(Dataset):
    """模拟图像数据集"""
    
    def __init__(self, num_samples, transform=None):
        self.num_samples = num_samples
        self.transform = transform
    
    def __getitem__(self, idx):
        # 创建随机图像
        img_array = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
        img = Image.fromarray(img_array)
        
        # 应用 transform
        if self.transform:
            img = self.transform(img)
        
        # 随机标签
        label = idx % 3  # 3 个类别
        
        return img, label
    
    def __len__(self):
        return self.num_samples

# 定义 transform
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# 创建数据集和 DataLoader
dataset = MockImageDataset(num_samples=10, transform=transform)
loader = DataLoader(dataset, batch_size=4, shuffle=False)

# 查看批量数据
print("图像数据的批量加载：\n")
for batch_idx, (images, labels) in enumerate(loader):
    print(f"Batch {batch_idx}:")
    print(f"  图像形状: {images.shape}")
    print(f"  标签形状: {labels.shape}")
    print(f"  标签内容: {labels}")
    print()

**关键理解：**

- Dataset 返回单个样本：`(C, H, W)`
- DataLoader 返回一个 batch：`(B, C, H, W)`
- B = batch_size

## 6. num_workers：多进程加载

In [ ]:
import time

class SlowDataset(Dataset):
    """模拟慢速数据加载"""
    
    def __init__(self, n):
        self.data = list(range(n))
    
    def __getitem__(self, idx):
        time.sleep(0.01)  # 模拟从硬盘读取的耗时
        return self.data[idx]
    
    def __len__(self):
        return len(self.data)

dataset = SlowDataset(100)

# num_workers=0（单进程）
print("测试 num_workers=0（单进程）:")
loader_0 = DataLoader(dataset, batch_size=10, num_workers=0)
start = time.time()
for batch in loader_0:
    pass
print(f"耗时: {time.time() - start:.2f} 秒\n")

# num_workers=2（多进程）
print("测试 num_workers=2（多进程）:")
loader_2 = DataLoader(dataset, batch_size=10, num_workers=2)
start = time.time()
for batch in loader_2:
    pass
print(f"耗时: {time.time() - start:.2f} 秒")

print("\n多进程加载可以显著提升速度！")

**num_workers 设置建议：**

```python
# 一般设置
num_workers = 4  # 或 8

# 根据 CPU 核心数
import os
num_workers = min(8, os.cpu_count())

# Windows 系统
num_workers = 0  # Windows 多进程可能有问题
```

**注意：** num_workers 太大会占用过多内存！

## 7. collate_fn：自定义批量拼接

### 7.1 默认 collate_fn 的行为

In [ ]:
# 默认情况下，DataLoader 如何拼接样本？

# 单个样本
sample1 = (torch.tensor([1, 2, 3]), 0)  # (data, label)
sample2 = (torch.tensor([4, 5, 6]), 1)
sample3 = (torch.tensor([7, 8, 9]), 0)

# 默认 collate_fn 做了什么？
# 1. 分离数据和标签
data_list = [sample1[0], sample2[0], sample3[0]]
label_list = [sample1[1], sample2[1], sample3[1]]

# 2. 堆叠成 batch
batch_data = torch.stack(data_list)
batch_labels = torch.tensor(label_list)

print("单个样本形状:", sample1[0].shape)
print("批量数据形状:", batch_data.shape)
print("批量标签形状:", batch_labels.shape)
print("\n批量数据:\n", batch_data)
print("批量标签:", batch_labels)

### 7.2 何时需要自定义 collate_fn？

1. **处理变长序列**（如文本）
2. **过滤无效样本**（返回 None 的样本）
3. **返回字典格式**
4. **复杂的数据结构**

### 7.3 示例 1：处理变长序列

In [ ]:
from torch.nn.utils.rnn import pad_sequence

class VariableLengthDataset(Dataset):
    """变长序列数据集"""
    
    def __init__(self):
        # 不同长度的序列
        self.sequences = [
            torch.tensor([1, 2, 3]),
            torch.tensor([4, 5]),
            torch.tensor([6, 7, 8, 9]),
            torch.tensor([10])
        ]
        self.labels = [0, 1, 0, 1]
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]
    
    def __len__(self):
        return len(self.sequences)

def collate_fn_pad(batch):
    """自定义 collate_fn：填充变长序列"""
    sequences, labels = zip(*batch)
    
    # 填充到相同长度
    sequences_padded = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=0
    )
    labels = torch.tensor(labels)
    
    return sequences_padded, labels

# 创建数据集
dataset = VariableLengthDataset()

# 使用自定义 collate_fn
loader = DataLoader(
    dataset,
    batch_size=2,
    collate_fn=collate_fn_pad
)

print("处理变长序列：\n")
for batch_idx, (sequences, labels) in enumerate(loader):
    print(f"Batch {batch_idx}:")
    print(f"  序列形状: {sequences.shape}")
    print(f"  序列内容:\n{sequences}")
    print(f"  标签: {labels}\n")

### 7.4 示例 2：返回字典格式

In [ ]:
class DictDataset(Dataset):
    """返回多个信息的数据集"""
    
    def __init__(self, n):
        self.n = n
    
    def __getitem__(self, idx):
        return {
            'image': torch.randn(3, 32, 32),
            'label': idx % 2,
            'index': idx,
            'path': f'img_{idx}.jpg'
        }
    
    def __len__(self):
        return self.n

def collate_fn_dict(batch):
    """处理字典格式的样本"""
    # batch 是一个列表，每个元素是一个字典
    # 需要将它转换为一个字典，每个键对应一个 batch
    
    images = torch.stack([item['image'] for item in batch])
    labels = torch.tensor([item['label'] for item in batch])
    indices = [item['index'] for item in batch]
    paths = [item['path'] for item in batch]
    
    return {
        'images': images,
        'labels': labels,
        'indices': indices,
        'paths': paths
    }

# 创建数据集和 DataLoader
dataset = DictDataset(6)
loader = DataLoader(
    dataset,
    batch_size=3,
    collate_fn=collate_fn_dict
)

print("字典格式的批量数据：\n")
for batch_idx, batch in enumerate(loader):
    print(f"Batch {batch_idx}:")
    print(f"  images 形状: {batch['images'].shape}")
    print(f"  labels: {batch['labels']}")
    print(f"  indices: {batch['indices']}")
    print(f"  paths: {batch['paths']}\n")

## 8. 训练循环中使用 DataLoader

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 模拟数据集
train_dataset = MockImageDataset(num_samples=100, transform=transform)
valid_dataset = MockImageDataset(num_samples=30, transform=transform)

# 创建 DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,      # 训练集打乱
    num_workers=0
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,     # 验证集不打乱
    num_workers=0
)

# 模拟训练循环
print("训练循环示例：\n")

num_epochs = 2
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    
    # 训练阶段
    print("  训练阶段:")
    for batch_idx, (images, labels) in enumerate(train_loader):
        # images: [batch_size, 3, 32, 32]
        # labels: [batch_size]
        
        # 前向传播、计算损失、反向传播
        # ...
        
        if batch_idx < 2:  # 只打印前 2 个 batch
            print(f"    Batch {batch_idx}: images {images.shape}, labels {labels.shape}")
    
    # 验证阶段
    print("  验证阶段:")
    for batch_idx, (images, labels) in enumerate(valid_loader):
        # 验证代码
        # ...
        
        if batch_idx < 1:
            print(f"    Batch {batch_idx}: images {images.shape}, labels {labels.shape}")
    
    print()

## 9. 常见问题与解决方案

### Q1: num_workers 应该设置为多少？

In [ ]:
import os

# 推荐设置
num_workers_recommended = 4  # 一般设置为 4-8

# 根据 CPU 核心数
num_workers_auto = min(8, os.cpu_count() or 1)

print(f"推荐设置: num_workers={num_workers_recommended}")
print(f"自动设置: num_workers={num_workers_auto}")
print(f"CPU 核心数: {os.cpu_count()}")

# Windows 系统
# num_workers = 0  # Windows 多进程可能有问题

### Q2: 什么时候使用 pin_memory？

In [ ]:
# 使用 GPU 训练时，建议开启
train_loader_gpu = DataLoader(
    train_dataset,
    batch_size=32,
    pin_memory=True  # 加快数据传输到 GPU
)

# 使用 CPU 训练时，不需要
train_loader_cpu = DataLoader(
    train_dataset,
    batch_size=32,
    pin_memory=False
)

print("pin_memory 的作用：")
print("- 将数据锁定在内存中")
print("- 加快 CPU → GPU 的数据传输")
print("- 仅在使用 GPU 时有效")

### Q3: shuffle 和 sampler 能同时使用吗？

In [ ]:
from torch.utils.data import RandomSampler

# ❌ 错误：不能同时使用
# loader = DataLoader(
#     dataset,
#     batch_size=32,
#     shuffle=True,
#     sampler=RandomSampler(dataset)
# )

# ✅ 正确：二选一
# 选择 1：使用 shuffle
loader1 = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

# 选择 2：使用 sampler
loader2 = DataLoader(
    dataset,
    batch_size=32,
    sampler=RandomSampler(dataset)
)

print("shuffle 和 sampler 互斥，只能选一个！")

## 10. 练习题

### 练习 1：观察 batch_size 的影响

创建一个数据集，尝试不同的 batch_size，观察输出形状

In [ ]:
# TODO: 创建数据集
# dataset = ...

# TODO: 尝试 batch_size = 1, 4, 16
# 观察每个 batch 的形状

### 练习 2：实现自定义 collate_fn

实现一个 collate_fn，过滤掉标签为 -1 的样本

In [ ]:
def collate_fn_filter(batch):
    """过滤掉标签为 -1 的样本"""
    # TODO: 实现过滤逻辑
    pass

# 测试代码
# class TestDataset(Dataset):
#     def __getitem__(self, idx):
#         return torch.randn(3, 32, 32), -1 if idx % 3 == 0 else idx % 2
#     def __len__(self):
#         return 10

# dataset = TestDataset()
# loader = DataLoader(dataset, batch_size=3, collate_fn=collate_fn_filter)
# for batch in loader:
#     print(batch)

## 11. 小结

### 核心要点

1. **DataLoader 的作用**：批量加载、打乱、多进程
2. **关键参数**：
   - `batch_size`：批量大小
   - `shuffle`：是否打乱（训练集 True，验证集 False）
   - `num_workers`：多进程数量
   - `drop_last`：是否丢弃最后不完整的 batch
   - `collate_fn`：自定义批量拼接函数

### 最佳实践

```python
# 训练集 DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,      # 打乱
    num_workers=4,     # 多进程
    pin_memory=True,   # GPU 加速
    drop_last=False    # 保留最后的 batch
)

# 验证集 DataLoader
valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,     # 不打乱
    num_workers=4,
    pin_memory=True
)
```

### 下一步

在下一个 notebook 中，我们将学习 **Sampler**，实现更灵活的数据采样策略！

---

## 📝 练习题

完成本章学习后，请尝试 [exercises.md](./exercises.md) 中的练习题来巩固知识。

**建议：**
- 先完成基础题，确保理解核心概念
- 进阶题帮助你深入掌握技巧
- 挑战题锻炼综合应用能力

💪 加油！实践是最好的学习方式！